In [ ]:
import torchvision.transforms as T
import torch
import torchvision.io as io
import matplotlib.pyplot as plt
from utils.EarlyStopping import EarlyStopping
from data_preprocessing.vision_dataset import imagenette2Dataset
from model.ViT import VisionTransformer
import tqdm as notebook_tqdm
import tqdm
from utils.train_val_fn import train_function, validation_function
%matplotlib inline

In [ ]:
image_tensor = io.read_image("imagenette2/train/n01440764/ILSVRC2012_val_00000293.JPEG")

In [ ]:
image_tensor = image_tensor.float() / 255.0

In [ ]:
print(image_tensor.shape) #Channels, Height, withd

# Rearrange dimensions from (C, H, W) to (H, W, C)
image_for_plot = image_tensor.permute(1, 2, 0)

plt.imshow(image_for_plot)
plt.title(f"Training example #{1}")
plt.axis('off')
plt.show()

In [ ]:
dataset = imagenette2Dataset(data_path="imagenette2", image_size=224, batch_size=32)

In [ ]:
train_loader, val_loader = dataset.create_dataloaders()

In [ ]:
for i, (images, labels) in enumerate(train_loader):
    print(f"the shape of the image(input) {images.shape}")
    print(f"the shape of the label(label) {labels.shape}")
    break

### Model training

In [ ]:
n_layers = 6
d_dimensionality = 256
mlp_size = 768
n_heads = 4
dropout = 0.1
patch_size = 16
input_channels = 3
image_size = 224
n_classes = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
vision_transformer = VisionTransformer(n_layers=n_layers,
                                       d_dimensionality=d_dimensionality,
                                       mlp_size=mlp_size,
                                       n_heads=n_heads,
                                       dropout=dropout,
                                       in_channels=input_channels,
                                       patch_size=patch_size,
                                       image_size=image_size,
                                       num_classes=n_classes,
                                       device=device)

In [ ]:
optimizer = torch.optim.AdamW(vision_transformer.parameters(), lr=3e-3, weight_decay=0.3)
lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, 
    T_max=50, 
    eta_min=1e-6
)

In [ ]:
print(f"Model parameters: {sum(p.numel() for p in vision_transformer.parameters())/1e6:.2f} M parameters")

In [ ]:
n_epochs = 50
n_num_epochs = 0
best_valid_loss = float('inf')
early_stopping = EarlyStopping(patience=5, mode="max" ,verbose=True, path='best_model.pt')
for epoch in tqdm.tqdm(range(n_epochs), desc="Training Progress"):
    
    train_loss = train_function(vision_transformer, train_loader, optimizer, device)
    
    val_loss, val_acc = validation_function(vision_transformer, val_loader, optimizer, device )
    
    print(f"Epoch [{epoch+1}/{n_epochs}] | Train Loss: {train_loss:.4f} | Valid Loss: {val_loss:.4f} | Valid Acc: {val_acc * 100:.2f}%")
    
    early_stopping(val_acc, vision_transformer)

    if early_stopping.early_stop:
        print("Finished because of the early Stopping")
        break 
    
    